In [0]:
dbutils.widgets.text("catalog", "dennis_schultz", "Catalog Name")
dbutils.widgets.text("schema", "vantage_workshop", "Schema Name")
dbutils.widgets.text("volume", "volume", "Volume to store user list")
dbutils.widgets.text("file_name", "file", "List of users")
dbutils.widgets.text("group", "group", "Name of group where users should be added")

# Unity Catalog
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
filename = dbutils.widgets.get("file_name")
GROUP_NAME = dbutils.widgets.get("group")

FILE_PATH = f"/Volumes/{catalog}/{schema}/{volume}/{filename}"

In [0]:
%pip install openpyxl -q

In [0]:
import pandas as pd
from databricks.sdk import WorkspaceClient, AccountClient
from databricks.sdk.service.iam import Patch, PatchOp, PatchSchema

# Workspace client — user provisioning (auto-authenticates on compute)
w = WorkspaceClient()


In [0]:
def find_group(group_name):
    '''
    Find a group by name using the WorkspaceClient.
    '''
    group = next(
        (g for g in w.groups.list(filter=f'displayName eq "{group_name}"'))
    )
    return group

In [0]:
def get_user(email, members):
    '''
    Find a user by email using the WorkspaceClient.
    If the user doesn't exist, attempt to provision a new user
    '''
    users = list(w.users.list(filter=f'userName eq "{email}"'))
    if users:
        members.append(users[0].id)
    else:
        try:
            new_user = w.users.create(user_name=email)
            members.append(new_user.id)
            print(f"Provisioned new workspace user: {email}")
        except Exception as e:
            print(f"⚠️ Could not provision {email}: {e}")
    return members

In [0]:
def add_members(members, group):
    '''
    Add members to a group using the workspace REST client.
    Must use the /api/2.0/account/scim/v2/Groups API to add the members.  
    The w.groups.patch() method is read-only for account-level groups and the account client requires account admin permissions
    '''
    for m in members:
        w.api_client.do( 
            "PATCH", 
            f"/api/2.0/account/scim/v2/Groups/{group.id}", 
            body={ 
                "Operations": [ { 
                    "op": "add", 
                    "value": {"members": [
                        {"value": m}
                    ]} 
                } ], 
                "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"], 
            },
        )    
    print(f"✅ Added {len(members)} users to '{GROUP_NAME}'")

In [0]:
# 1.  Find the group
group =find_group(GROUP_NAME)
print(group)

# 2. Read user emails from Excel
df = pd.read_excel(FILE_PATH)
emails = df["Email"].dropna().tolist()
print(emails)

# 3. Look up each user; provision to workspace if not found
members = []
for email in emails:
    members=get_user(email, members)
print(members)

# 4. PATCH the group via account client
if members:
    add_members(members, group)
else:
    print(f"No users to add to '{GROUP_NAME}'")